In [1]:
import os

In [2]:
%pwd

'c:\\Users\\siddh\\Desktop\\Study Material\\Data Science\\0.Projects.py\\5.Poultry Disease Detection System\\Poultry-Diseases-Detection\\research'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\siddh\\Desktop\\Study Material\\Data Science\\0.Projects.py\\5.Poultry Disease Detection System\\Poultry-Diseases-Detection'

In [4]:
import tensorflow as tf

In [5]:
model=tf.keras.models.load_model("artifacts/training/model.h5")

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [7]:
from src.PoultryDiseasesCNN.constants import *
from src.PoultryDiseasesCNN.utils.common import read_yaml, create_directories, save_json

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=Path("artifacts/data_ingestion/unzipped_data/Chicken_Fecal_Images"),
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [9]:
from urllib.parse import urlparse

In [12]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(str(path))  # Ensure it's a string
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.scores = self.model.evaluate(self.valid_generator)  # Store scores for later
    
    def save_score(self):
        print(f"loss: {self.scores[0]} accuracy: {self.scores[1]}")
        scores_dict = {"loss": self.scores[0], "accuracy": self.scores[1]}
        save_json(path=Path("scores.json"), data=scores_dict)

In [13]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()
    
except Exception as e:
    raise e

[2025-10-18 16:03:58,431: INFO: common]: yaml file: config\config.yaml loaded successfully
[2025-10-18 16:03:58,437: INFO: common]: yaml file: params.yaml loaded successfully
[2025-10-18 16:03:58,443: INFO: common]: created directory at: artifacts
Found 2041 images belonging to 4 classes.
128/128 [==============================] - 168s 1s/step - loss: 0.9451 - accuracy: 0.7237
loss: 0.9450693726539612 accuracy: 0.7236648797988892
[2025-10-18 16:06:47,867: INFO: common]: json file saved at: scores.json
